# Dataset Expansion: 50 → 74 Contracts

**Goal:** Expand dataset from 50 (4 failed) to ~74 valid contracts (14–15 per domain)  
**Strategy:**
- Primary source: `docs/metadata.csv` (10 pre-vetted contracts per domain)  
- Supplement: old `contracts_dataset/` (pick best 4–5 non-overlapping per domain)  
- Target: DeFi=14, NFT=15, Token=15, Governance=15, Utility=15  
- Simple contract confirmed: Multicall2 (LOC=66) from metadata.csv

**Run cells in order. Requires ETHERSCAN_API_KEY in .env**

In [1]:
import csv, json, os, re, sys, time, shutil
from pathlib import Path
from dotenv import load_dotenv
import requests

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
API_KEY = os.getenv('ETHERSCAN_API_KEY', '')
if not API_KEY:
    raise ValueError('ETHERSCAN_API_KEY tidak ditemukan di .env')

print('ROOT:', ROOT)
print('API_KEY:', API_KEY[:6] + '...')

ROOT: C:\Users\Ridho\project\KBJ\gas_optimization
API_KEY: SNQRM9...


## 1. Verify Backups Exist

In [2]:
backup_meta = ROOT / 'contracts_metadata_backup.json'
backup_results = ROOT / 'results' / 'backup'

assert backup_meta.exists(), 'Run: cp contracts_metadata.json contracts_metadata_backup.json'
assert backup_results.exists(), 'Run: cp -r results/. results/backup/'

print('Backup metadata:', backup_meta.name, '✓')
print('Backup results:', backup_results, '✓')
print('Backup results files:', len(list(backup_results.glob('*'))))

Backup metadata: contracts_metadata_backup.json ✓
Backup results: C:\Users\Ridho\project\KBJ\gas_optimization\results\backup ✓
Backup results files: 18


## 2. Define Target Contract List

In [3]:
# Load old metadata
old_meta = json.load(open(ROOT / 'contracts_metadata.json'))
old_by_addr = {x['alamat'].lower(): x for x in old_meta}

# Load new metadata.csv
with open(ROOT / 'docs/metadata.csv') as f:
    csv_rows = list(csv.DictReader(f))
csv_by_addr = {r['address'].lower(): r for r in csv_rows}

# --- Old contracts to keep (non-overlap with csv, compile_ok=True) ---
# Selected: best-known / most interesting per domain
OLD_KEEP = {
    # DeFi: all 4 non-overlaps (can only reach 14, not 15)
    '0x7a250d5630b4cf539739df2c5dacb4c659f2488d': ('DeFi',  'contracts_dataset/DeFi_02_UniswapV2Router02.sol'),
    '0x6b175474e89094c44da98b954eedeac495271d0f': ('DeFi',  'contracts_dataset/DeFi_03_Dai.sol'),
    '0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9': ('DeFi',  'contracts_dataset/DeFi_06_InitializableAdminUpgradeabilityProxy.sol'),
    '0xae7ab96520de3a18e5e111b5eaab095312d7fe84': ('DeFi',  'contracts_dataset/DeFi_08_AppProxyUpgradeable.sol'),
    # NFT: 5 best-known from 8 non-overlaps
    '0x60e4d786628fea6478f785a6d7e704777c86a7c6': ('NFT',   'contracts_dataset/NFT_03_MutantApeYachtClub.sol'),
    '0x49cf6f5d44e70224e2e23fdcdd2c053f30ada28b': ('NFT',   'contracts_dataset/NFT_04_CloneX.sol'),
    '0xed5af388653567af2f388e6224dc7c4b3241c544': ('NFT',   'contracts_dataset/NFT_06_Azuki.sol'),
    '0x8a90cab2b38dba80c64b7734e58ee1db38b8992e': ('NFT',   'contracts_dataset/NFT_07_Doodles.sol'),
    '0x7bd29408f11d2bfc23c34f18275bbf23bb716bc7': ('NFT',   'contracts_dataset/NFT_08_Meebits.sol'),
    # Token: all 5 non-overlaps
    '0x111111111117dc0aa78b770fa6a738034120c302': ('Token', 'contracts_dataset/Token_05_OneInch.sol'),
    '0xba100000625a3754423978a60c9317c58a424e3d': ('Token', 'contracts_dataset/Token_07_BalancerGovernanceToken.sol'),
    '0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f': ('Token', 'contracts_dataset/Token_08_ProxyERC20.sol'),
    '0x4d224452801aced8b2f0aebe155379bb5d594381': ('Token', 'contracts_dataset/Token_09_SimpleToken.sol'),
    '0x6dea81c8171d0ba574754ef6f8b412f2ed88c54d': ('Token', 'contracts_dataset/Token_10_LQTYToken.sol'),
    # Governance: 5 best from 8 non-overlaps
    '0xc0da02939e1441f497fd74f78ce7decb17b66529': ('Governance', 'contracts_dataset/Governance_02_GovernorBravoDelegator.sol'),
    '0xec568fffba86c094cf06b22134b23074dfe2252c': ('Governance', 'contracts_dataset/Governance_04_AaveGovernanceV2.sol'),
    '0x690e775361ad66d1c4a25d89da9fcd639f5198ed': ('Governance', 'contracts_dataset/Governance_05_Governor.sol'),
    '0x6f3e6272a167e8accb32072d08e0957f9c79223d': ('Governance', 'contracts_dataset/Governance_09_NounsDAOProxy.sol'),
    '0xdbd27635a534a3d3169ef0498beb56fb9c937489': ('Governance', 'contracts_dataset/Governance_06_GovernorAlpha.sol'),
    # Utility: 5 best from 8 non-overlaps
    '0xa646e29877d52b9e2de457eca09c724ff16d0a2b': ('Utility', 'contracts_dataset/Utility_03_MultiSigWallet.sol'),
    '0x19c0976f590d67707e62397c87829d896dc0f1f1': ('Utility', 'contracts_dataset/Utility_04_Jug.sol'),
    '0x1f98431c8ad98523631ae4a59f267346ea31f984': ('Utility', 'contracts_dataset/Utility_07_UniswapV3Factory.sol'),
    '0x68b3465833fb72a70ecdf485e0e4c7bd8665fc45': ('Utility', 'contracts_dataset/Utility_09_SwapRouter02.sol'),
    '0xc36442b4a4522e871399cd717abdd847ab11fe88': ('Utility', 'contracts_dataset/Utility_10_NonfungiblePositionManager.sol'),
}

print(f'CSV contracts: {len(csv_rows)}')
print(f'Old keep contracts: {len(OLD_KEEP)}')
print()

from collections import Counter
domain_count_csv = Counter(r['domain'] for r in csv_rows)
domain_count_old = Counter(v[0] for v in OLD_KEEP.values())
for domain in ['DeFi','NFT','Token','Governance','Utility']:
    total = domain_count_csv[domain] + domain_count_old[domain]
    print(f'  {domain:12s}: {domain_count_csv[domain]} csv + {domain_count_old[domain]} old = {total}')

CSV contracts: 50
Old keep contracts: 24

  DeFi        : 10 csv + 4 old = 14
  NFT         : 10 csv + 5 old = 15
  Token       : 10 csv + 5 old = 15
  Governance  : 10 csv + 5 old = 15
  Utility     : 10 csv + 5 old = 15


## 3. Download New Contracts from metadata.csv

In [4]:
def fetch_source(address):
    url = (f'https://api.etherscan.io/v2/api?chainid=1'
           f'&module=contract&action=getsourcecode'
           f'&address={address}&apikey={API_KEY}')
    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=15)
            data = resp.json()
            break
        except Exception as e:
            if attempt == 2: return None, None, str(e)
            time.sleep(2**attempt)
    if data.get('status') != '1': return None, None, data.get('message','API error')
    r = data['result'][0]
    src = r.get('SourceCode','')
    name = r.get('ContractName','')
    if not src: return None, None, 'not verified'
    # flatten {{json}} format
    if src.startswith('{{'):
        try:
            fj = json.loads(src[1:-1])
            parts = []
            for path, content in fj['sources'].items():
                parts.append(f'// === File: {path} ===')
                parts.append(content['content'])
            src = '\n\n'.join(parts)
        except Exception: pass
    return src, name, None

downloaded, skipped, failed_dl = 0, 0, []

for row in csv_rows:
    dest = ROOT / row['source_file']
    dest.parent.mkdir(parents=True, exist_ok=True)
    
    if dest.exists():
        skipped += 1
        continue
    
    # For proxy: try implementation address first
    is_proxy = row.get('proxy','0') == '1'
    impl = row.get('implementation','').strip()
    src = None
    for addr in ([impl, row['address']] if (is_proxy and impl) else [row['address']]):
        if not addr: continue
        raw_src, _, err = fetch_source(addr)
        if raw_src:
            src = raw_src
            break
        time.sleep(0.3)
    
    if not src:
        print(f'  FAIL {row["contract_name"]:35s} {err}')
        failed_dl.append(row)
        time.sleep(0.25)
        continue
    
    dest.write_text(src, encoding='utf-8')
    print(f'  DL   {row["contract_name"]:35s} LOC={len(src.splitlines()):5d}')
    downloaded += 1
    time.sleep(0.25)

print(f'\nDownloaded: {downloaded}, Skipped (exists): {skipped}, Failed: {len(failed_dl)}')


Downloaded: 0, Skipped (exists): 50, Failed: 0


## 4. Compile-Validate All New Contracts

In [5]:
import solcx, tempfile
from packaging.version import Version as PkgVersion

SOLC_MAP = {'0.4':'0.4.26','0.5':'0.5.17','0.6':'0.6.12','0.7':'0.7.6','0.8':'0.8.20'}

def detect_ver(src):
    m = re.search(r'pragma solidity\s*=?\s*(\d+\.\d+\.\d+)\s*;', src)
    if m: return m.group(1)
    m = re.search(r'pragma solidity\s+[>=^<~]*\s*(\d+\.\d+)', src)
    if not m: return '0.8.20'
    mm = '.'.join(m.group(1).split('.')[:2])
    return SOLC_MAP.get(mm, '0.8.20')

def dedup_pragma(src):
    seen, lines = False, []
    for line in src.splitlines():
        if re.match(r'\s*pragma solidity', line):
            if not seen: lines.append(line); seen = True
        else:
            lines.append(line)
    return '\n'.join(lines)

def parse_multifile(src):
    parts = re.split(r'// === File: (.+?) ===', src)
    if len(parts) <= 1: return None
    files = {}
    for i in range(1, len(parts)-1, 2):
        files[parts[i].strip()] = parts[i+1]
    return files or None

def try_compile(src, ver):
    files = parse_multifile(src)
    if PkgVersion(ver) < PkgVersion('0.4.11'): ver = '0.4.26'
    installed = [str(v) for v in solcx.get_installed_solc_versions()]
    if ver not in installed:
        print(f'    installing solc {ver}...')
        solcx.install_solc(ver)
    if files:
        try:
            inp = {"language":"Solidity",
                   "sources":{n:{"content":c} for n,c in files.items()},
                   "settings":{"optimizer":{"enabled":False},
                               "outputSelection":{"*":{"":["ast"]}}}}
            solcx.compile_standard(inp, solc_version=ver)
            return True
        except Exception: pass
        try:
            with tempfile.TemporaryDirectory() as td:
                tdp = Path(td)
                for name, content in files.items():
                    fp2 = tdp/name; fp2.parent.mkdir(parents=True,exist_ok=True)
                    fp2.write_text(content, encoding='utf-8')
                solcx.compile_files([str(tdp/next(iter(files)))],
                    output_values=['ast'], solc_version=ver, optimize=False, allow_paths=[td])
                return True
        except Exception: pass
    else:
        try:
            solcx.compile_source(dedup_pragma(src), output_values=['ast'],
                                  solc_version=ver, optimize=False)
            return True
        except Exception: pass
        try:
            stripped = re.sub(r'^\s*import\s+[^;]+;','',src,flags=re.MULTILINE)
            solcx.compile_source(dedup_pragma(stripped), output_values=['ast'],
                                  solc_version=ver, optimize=False)
            return True
        except Exception: pass
    return False

csv_results = []
print('Validating CSV contracts...')
for row in csv_rows:
    fp = ROOT / row['source_file']
    if not fp.exists():
        print(f'  MISSING {row["contract_name"]}')
        csv_results.append({**row, 'compile_ok': False, 'actual_loc': 0})
        continue
    src = fp.read_text(encoding='utf-8')
    actual_loc = len(src.splitlines())
    ver = detect_ver(src)
    ok = try_compile(src, ver)
    status = 'OK  ' if ok else 'FAIL'
    print(f'  [{status}] {row["domain"]:12s} {row["contract_name"]:35s} LOC={actual_loc:5d} solc={ver}')
    csv_results.append({**row, 'compile_ok': ok, 'actual_loc': actual_loc, 'solc_ver': ver})

print(f'\nCompile OK: {sum(1 for r in csv_results if r["compile_ok"])}/{len(csv_results)}')

Validating CSV contracts...


  [OK  ] DeFi         WETH9                               LOC= 1511 solc=0.4.26


  [OK  ] DeFi         UniswapV2Factory                    LOC=  987 solc=0.5.16


  [OK  ] DeFi         CErc20Delegator                     LOC= 2293 solc=0.8.20


  [OK  ] DeFi         CEther                              LOC= 2540 solc=0.5.17
  [OK  ] DeFi         UniswapV2Pair                       LOC=  991 solc=0.5.16


...

  [OK  ] Governance   AdminUpgradeabilityProxy            LOC= 1589 solc=0.5.17
  [OK  ] Utility      GnosisSafe                          LOC= 1132 solc=0.7.6


  [OK  ] Utility      ENSRegistryWithFallback             LOC=  565 solc=0.4.26
  [OK  ] Utility      ERC1820Registry                     LOC=  433 solc=0.5.3
  [OK  ] Utility      Multicall2                          LOC=  151 solc=0.5.17


  [OK  ] Utility      WyvernExchange                      LOC= 3663 solc=0.4.26
  [OK  ] Utility      GnosisSafeProxyFactory              LOC=  173 solc=0.7.6
  [OK  ] Utility      DSProxyFactory                      LOC=  431 solc=0.4.26


  [OK  ] Utility      ReverseRegistrar                    LOC=  434 solc=0.8.20
  [FAIL] Utility      PublicResolver                      LOC=  897 solc=0.4.26


  [OK  ] Utility      WyvernProxyRegistry                 LOC=  901 solc=0.4.26

Compile OK: 49/50


## 5. Merge Into Expanded Metadata

In [6]:
expanded = []
seen_addrs = set()

# Part A: All CSV contracts (compile_ok ones)
for r in csv_results:
    addr = r['address'].lower()
    seen_addrs.add(addr)
    complexity = r.get('complexity', '')
    # Recompute actual complexity from actual_loc
    loc = r.get('actual_loc', int(r.get('loc',0)))
    if loc < 100:   complexity = 'Simple'
    elif loc < 500: complexity = 'Medium'
    else:           complexity = 'Complex'
    expanded.append({
        'alamat':     r['address'],
        'domain':     r['domain'],
        'nama':       r['contract_name'],
        'file':       str(ROOT / r['source_file']),
        'loc':        loc,
        'complexity': complexity,
        'compile_ok': r['compile_ok'],
        'solc_ver':   r.get('solc_ver', r.get('compiler_version','').split('+')[0].lstrip('v')),
        'is_proxy':   r.get('proxy','0') == '1',
        'source':     'csv',
    })

# Part B: Selected old contracts
old_by_addr_lower = {x['alamat'].lower(): x for x in old_meta}
for addr_lower, (domain, rel_file) in OLD_KEEP.items():
    if addr_lower in seen_addrs:
        print(f'SKIP (already in csv): {addr_lower[:14]}')
        continue
    old = old_by_addr_lower.get(addr_lower)
    if not old:
        print(f'MISSING in old meta: {addr_lower[:14]}')
        continue
    seen_addrs.add(addr_lower)
    fp = ROOT / rel_file
    if not fp.exists():
        print(f'FILE MISSING: {rel_file}')
        continue
    loc = len(fp.read_text(encoding='utf-8').splitlines())
    complexity = 'Simple' if loc < 100 else 'Medium' if loc < 500 else 'Complex'
    expanded.append({
        'alamat':     old['alamat'],
        'domain':     domain,
        'nama':       old['nama'],
        'file':       str(fp),
        'loc':        loc,
        'complexity': complexity,
        'compile_ok': old.get('compile_ok', False),
        'solc_ver':   old.get('solc_ver', ''),
        'is_proxy':   old.get('is_proxy', False),
        'source':     'old',
    })

# Summary
from collections import Counter
print(f'\nTotal contracts: {len(expanded)}')
print(f'Compile OK: {sum(1 for x in expanded if x["compile_ok"])}')
print()
domain_counts = Counter(x['domain'] for x in expanded if x['compile_ok'])
for d, n in domain_counts.items():
    print(f'  {d:12s}: {n}')
print()
complexity_counts = Counter(x['complexity'] for x in expanded if x['compile_ok'])
for c, n in complexity_counts.items():
    print(f'  {c:10s}: {n}')

# Save
out = ROOT / 'contracts_metadata_expanded.json'
with open(out, 'w') as f:
    json.dump(expanded, f, indent=2, ensure_ascii=False)
print(f'\nSaved: {out}')


Total contracts: 74
Compile OK: 73

  DeFi        : 14
  NFT         : 15
  Token       : 15
  Governance  : 15
  Utility     : 14

  Complex   : 58
  Medium    : 15

Saved: C:\Users\Ridho\project\KBJ\gas_optimization\contracts_metadata_expanded.json


## 6. Run Detectors on New Contracts

Run the 6 detectors on contracts that are NOT yet in `pekan2_detector_results.json`

In [7]:
from src.detectors.redundant_sload import detect as detect_rsload
from src.detectors.unoptimized_loop import detect as detect_loop
from src.detectors.string_vs_bytes32 import detect as detect_str
from src.detectors.public_vs_external import detect as detect_pub
from src.detectors.unchecked_arithmetic import detect as detect_arith
from src.detectors.dead_code import detect as detect_dead
from src.ast_parser import generate_ast

DETECTORS = [
    ('redundant_sload',      detect_rsload),
    ('unoptimized_loop',     detect_loop),
    ('string_vs_bytes32',    detect_str),
    ('public_vs_external',   detect_pub),
    ('unchecked_arithmetic', detect_arith),
    ('dead_code',            detect_dead),
]

# Load existing results — format: flat dict with integer counts per detector
results_path = ROOT / 'results' / 'pekan2_detector_results.json'
existing = json.load(open(results_path)) if results_path.exists() else []

# Dedup by file basename (unique across both old and new naming schemes)
done_files = {r['file'] for r in existing}
print(f'Existing results: {len(existing)} contracts already analyzed')

# Only analyze contracts whose file basename is not yet in results
to_analyze = []
for x in expanded:
    if not x['compile_ok']:
        continue
    basename = Path(x['file']).name
    if basename in done_files:
        print(f'  skip (already done): {x["nama"]}')
    else:
        to_analyze.append(x)

print(f'New contracts to analyze: {len(to_analyze)}')

Existing results: 50 contracts already analyzed
  skip (already done): UniswapV2Router02
  skip (already done): Dai
  skip (already done): InitializableAdminUpgradeabilityProxy
  skip (already done): AppProxyUpgradeable
  skip (already done): MutantApeYachtClub
  skip (already done): CloneX
  skip (already done): Azuki
  skip (already done): Doodles
  skip (already done): Meebits
  skip (already done): OneInch
  skip (already done): BalancerGovernanceToken
  skip (already done): ProxyERC20
  skip (already done): SimpleToken
  skip (already done): LQTYToken
  skip (already done): GovernorBravoDelegator
  skip (already done): AaveGovernanceV2
  skip (already done): Governor
  skip (already done): NounsDAOProxy
  skip (already done): GovernorAlpha
  skip (already done): MultiSigWallet
  skip (already done): Jug
  skip (already done): UniswapV3Factory
  skip (already done): SwapRouter02
  skip (already done): NonfungiblePositionManager
New contracts to analyze: 49


In [8]:
import time as _time
new_results = []

for i, contract in enumerate(to_analyze):
    fp = Path(contract['file'])
    if not fp.exists():
        print(f'  [{i+1}/{len(to_analyze)}] SKIP (file missing): {contract["nama"]}')
        continue

    t0 = _time.perf_counter()

    try:
        ast_data = generate_ast(str(fp))  # pass filepath, not source string
    except Exception as e:
        print(f'  [{i+1}/{len(to_analyze)}] AST FAIL {contract["nama"]}: {e}')
        continue

    if ast_data is None:
        print(f'  [{i+1}/{len(to_analyze)}] AST NONE {contract["domain"]:12s} {contract["nama"]}')
        continue

    counts = {}
    for det_name, det_fn in DETECTORS:
        try:
            counts[det_name] = len(det_fn(ast_data))
        except Exception:
            counts[det_name] = 0

    elapsed = _time.perf_counter() - t0
    total_f = sum(counts.values())

    print(f'  [{i+1:2d}/{len(to_analyze)}] {contract["domain"]:12s} '
          f'{contract["nama"]:35s} findings={total_f:3d} t={elapsed:.3f}s')

    record = {
        'file':       Path(contract['file']).name,
        'nama':       contract['nama'],
        'alamat':     contract['alamat'],
        'domain':     contract['domain'],
        'complexity': contract['complexity'],
        'loc':        contract['loc'],
        'compile_ok': True,
        'analysis_time': elapsed,
        'source':     contract.get('source', 'csv'),
    }
    record.update(counts)
    new_results.append(record)

print(f'\nAnalyzed {len(new_results)} new contracts')

  [ 1/49] DeFi         WETH9                               findings=  9 t=0.073s


  [ 2/49] DeFi         UniswapV2Factory                    findings= 20 t=0.278s


  [ 3/49] DeFi         CErc20Delegator                     findings=  6 t=0.389s


  [ 4/49] DeFi         CEther                              findings= 49 t=0.567s


  [ 5/49] DeFi         UniswapV2Pair                       findings= 14 t=0.248s


...

  [39/49] Governance   Tribe                               findings=  7 t=0.198s
  [40/49] Governance   AdminUpgradeabilityProxy            findings= 10 t=0.203s


  [41/49] Utility      GnosisSafe                          findings=  2 t=0.267s
  [42/49] Utility      ENSRegistryWithFallback             findings=  9 t=0.132s


  [43/49] Utility      ERC1820Registry                     findings=  2 t=0.108s
  [44/49] Utility      Multicall2                          findings= 10 t=0.081s


  [45/49] Utility      WyvernExchange                      findings= 55 t=0.638s
  [46/49] Utility      GnosisSafeProxyFactory              findings=  0 t=0.079s
  [47/49] Utility      DSProxyFactory                      findings= 11 t=0.106s


  [48/49] Utility      ReverseRegistrar                    findings=  2 t=0.092s
  [49/49] Utility      WyvernProxyRegistry                 findings= 31 t=0.123s

Analyzed 49 new contracts


In [9]:
# Merge with existing and save
all_results = existing + new_results

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f'Total results saved: {len(all_results)} contracts')
print(f'  Existing: {len(existing)}, New: {len(new_results)}')

det_names = ['redundant_sload','unoptimized_loop','string_vs_bytes32',
             'public_vs_external','unchecked_arithmetic','dead_code']
totals = {d: sum(r.get(d, 0) for r in all_results) for d in det_names}
grand = sum(totals.values())
print(f'\nTotal findings: {grand}')
for d, n in totals.items():
    print(f'  {d:25s}: {n}')

Total results saved: 99 contracts
  Existing: 50, New: 49

Total findings: 2123
  redundant_sload          : 776
  unoptimized_loop         : 7
  string_vs_bytes32        : 310
  public_vs_external       : 860
  unchecked_arithmetic     : 10
  dead_code                : 160


## 7. Summary and Next Steps

In [10]:
expanded_meta = json.load(open(ROOT / 'contracts_metadata_expanded.json'))
valid = [x for x in expanded_meta if x['compile_ok']]

print('='*60)
print('DATASET EXPANSION COMPLETE')
print('='*60)
print(f'Total contracts: {len(expanded_meta)}')
print(f'Compile OK:      {len(valid)}')
print()

print('Per domain (compile_ok only):')
for d in ['DeFi','NFT','Token','Governance','Utility']:
    contracts = [x for x in valid if x['domain'] == d]
    simple  = sum(1 for x in contracts if x['complexity']=='Simple')
    medium  = sum(1 for x in contracts if x['complexity']=='Medium')
    complex_ = sum(1 for x in contracts if x['complexity']=='Complex')
    print(f'  {d:12s}: {len(contracts):2d}  (Simple={simple}, Medium={medium}, Complex={complex_})')

print()
print('Complexity distribution:')
for c in ['Simple','Medium','Complex']:
    n = sum(1 for x in valid if x['complexity']==c)
    print(f'  {c:10s}: {n}')

print()
print('Simple contracts (<100 LOC):')
for x in valid:
    if x['complexity'] == 'Simple':
        print(f'  {x["nama"]:35s} LOC={x["loc"]:3d} {x["domain"]} {x["alamat"][:14]}')

print()
print('Next: Run pekan4 experiment on expanded dataset')
print('      Use contracts_metadata_expanded.json as source')

DATASET EXPANSION COMPLETE
Total contracts: 74
Compile OK:      73

Per domain (compile_ok only):
  DeFi        : 14  (Simple=0, Medium=1, Complex=13)
  NFT         : 15  (Simple=0, Medium=1, Complex=14)
  Token       : 15  (Simple=0, Medium=1, Complex=14)
  Governance  : 15  (Simple=0, Medium=6, Complex=9)
  Utility     : 14  (Simple=0, Medium=6, Complex=8)

Complexity distribution:
  Simple    : 0
  Medium    : 15
  Complex   : 58

Simple contracts (<100 LOC):

Next: Run pekan4 experiment on expanded dataset
      Use contracts_metadata_expanded.json as source
